# R², Statistical Inference, and the Bridge to Classification

**Last session:** how does linear regression find the best line?
**This session:** we found the best line — but how do we know whether that line is actually *useful*, and whether individual variables really contribute?
**Next session:** what if the thing we want to predict is not a number, but a category such as 0/1?

Let's start by bringing back exactly the model we ended on last time.

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

pd.set_option("display.precision", 3)

<details>
<summary>Show code</summary>

```python
scores_1d = pd.read_csv("../data/study_scores_1d.csv")

model = LinearRegression()
model.fit(scores_1d[["hours_studied"]], scores_1d["exam_score"])
slope = model.coef_[0]
intercept = model.intercept_

predictions = slope * scores_1d["hours_studied"] + intercept
residuals = scores_1d["exam_score"] - predictions
sse = float((residuals ** 2).sum())

print(f"Best line: y = {slope:.2f}x + {intercept:.2f}")
print(f"SSE: {sse:.1f}")
```

</details>

In [2]:
scores_1d = pd.read_csv("../data/study_scores_1d.csv")

model = LinearRegression()
model.fit(scores_1d[["hours_studied"]], scores_1d["exam_score"])
slope = model.coef_[0]
intercept = model.intercept_

predictions = slope * scores_1d["hours_studied"] + intercept
residuals = scores_1d["exam_score"] - predictions
sse = float((residuals ** 2).sum())

print(f"Best line: y = {slope:.2f}x + {intercept:.2f}")
print(f"SSE: {sse:.1f}")

Best line: y = 5.83x + 34.06
SSE: 239.0


Last time, our whole objective was to find `slope` and `intercept` that minimized:

$$SSE = \sum_i (y_i - \hat y_i)^2$$

Now here's the question that's been quietly unanswered this whole time:

> **Suppose I tell you our best model has the SSE printed above. Is that good?**

Take a moment. Can you actually answer that with just the number above?

> ### 🙋 Ask the class
>
> - Is an SSE of this size good or bad? What would you need to know to answer that?

> ### 🧑‍🏫 Instructor note
>
> Students almost never can answer this confidently, and that's the point. SSE has units (squared exam-score
> points here) and no fixed scale — "420" means nothing until we compare it to something.

## Why SSE Alone Is Hard to Interpret

Here's why a raw SSE number is so hard to judge.

<details>
<summary>Show code</summary>

```python
comparison_figure = go.Figure()
comparison_figure.add_trace(
    go.Bar(
        x=["Dataset A<br>(scores 40-90)", "Dataset B<br>(scores 400-900)"],
        y=[400, 400],
        marker=dict(color=["#2563eb", "#dc2626"]),
        text=["SSE = 400", "SSE = 400"],
        textposition="outside",
    )
)
comparison_figure.update_layout(
    title="Same SSE, Very Different Problems",
    yaxis_title="SSE",
    template="plotly_white",
    width=650,
    height=400,
)
comparison_figure.show()
```

</details>

In [3]:
comparison_figure = go.Figure()
comparison_figure.add_trace(
    go.Bar(
        x=["Dataset A<br>(scores 40-90)", "Dataset B<br>(scores 400-900)"],
        y=[400, 400],
        marker=dict(color=["#2563eb", "#dc2626"]),
        text=["SSE = 400", "SSE = 400"],
        textposition="outside",
    )
)
comparison_figure.update_layout(
    title="Same SSE, Very Different Problems",
    yaxis_title="SSE",
    template="plotly_white",
    width=650,
    height=400,
)
comparison_figure.show()

Dataset A's scores range roughly 40-90. Dataset B's scores range roughly 400-900 — ten times the scale. An SSE of 400 is a very different achievement in each case.

Here's a second problem, even when the scale is the same:

```text
Model A SSE = 300
Model B SSE = 500
```

Model A looks better — but what if Model A's dataset was already easy (all points nearly identical), while Model B's dataset was inherently noisy and hard to predict at all?

> **We need some reference point.** That's exactly what R² gives us.

## The Dumbest Possible Baseline

Before we give any formula, let's build the simplest model imaginable — one that completely ignores `hours_studied`.

> If I know absolutely nothing about a student except everyone's historical scores, one reasonable prediction is simply the *average* score.

<details>
<summary>Show code</summary>

```python
baseline_prediction = scores_1d["exam_score"].mean()
baseline_predictions = np.full(len(scores_1d), baseline_prediction)

baseline_figure = go.Figure()
baseline_figure.add_trace(
    go.Scatter(
        x=scores_1d["hours_studied"], y=scores_1d["exam_score"], mode="markers",
        marker=dict(size=11, color="#2563eb"), name="actual students",
    )
)
baseline_figure.add_hline(y=baseline_prediction, line=dict(color="#f59e0b", width=3, dash="dash"))
baseline_figure.add_trace(
    go.Scatter(x=[None], y=[None], mode="lines", line=dict(color="#f59e0b", width=3, dash="dash"),
               name=f"baseline: predict mean ({baseline_prediction:.1f})")
)
baseline_figure.update_layout(
    title="The Baseline: Always Predict the Mean",
    xaxis_title="Hours studied", yaxis_title="Exam score",
    template="plotly_white", width=750, height=500,
)
baseline_figure.show()
```

</details>

In [4]:
baseline_prediction = scores_1d["exam_score"].mean()
baseline_predictions = np.full(len(scores_1d), baseline_prediction)

baseline_figure = go.Figure()
baseline_figure.add_trace(
    go.Scatter(
        x=scores_1d["hours_studied"], y=scores_1d["exam_score"], mode="markers",
        marker=dict(size=11, color="#2563eb"), name="actual students",
    )
)
baseline_figure.add_hline(y=baseline_prediction, line=dict(color="#f59e0b", width=3, dash="dash"))
baseline_figure.add_trace(
    go.Scatter(x=[None], y=[None], mode="lines", line=dict(color="#f59e0b", width=3, dash="dash"),
               name=f"baseline: predict mean ({baseline_prediction:.1f})")
)
baseline_figure.update_layout(
    title="The Baseline: Always Predict the Mean",
    xaxis_title="Hours studied", yaxis_title="Exam score",
    template="plotly_white", width=750, height=500,
)
baseline_figure.show()

This baseline's total squared error has its own name:

$$SST = \sum_i (y_i - \bar y)^2$$

("SST" = **total** sum of squares — how wrong we'd be if we completely ignored `hours_studied` and predicted the mean for everybody.)

In [5]:
sst = float(((scores_1d["exam_score"] - baseline_prediction) ** 2).sum())
print(f"SST (baseline error): {sst:.1f}")

SST (baseline error): 2868.6


## Comparing the Two Models — and Deriving R²

Same scatter, two candidate models: the flat baseline, and our fitted regression line. Residuals drawn for both.

<details>
<summary>Show code</summary>

```python
line_x = np.linspace(scores_1d["hours_studied"].min() - 0.5, scores_1d["hours_studied"].max() + 0.5, 50)

compare_figure = go.Figure()
compare_figure.add_trace(
    go.Scatter(x=scores_1d["hours_studied"], y=scores_1d["exam_score"], mode="markers",
               marker=dict(size=11, color="#2563eb"), name="actual")
)
compare_figure.add_hline(y=baseline_prediction, line=dict(color="#f59e0b", width=3, dash="dash"))
compare_figure.add_trace(
    go.Scatter(x=[None], y=[None], mode="lines", line=dict(color="#f59e0b", width=3, dash="dash"), name="baseline (mean)")
)
compare_figure.add_trace(
    go.Scatter(x=line_x, y=slope * line_x + intercept, mode="lines",
               line=dict(color="#dc2626", width=3), name="regression line")
)
for xi, yi, pi in zip(scores_1d["hours_studied"], scores_1d["exam_score"], predictions):
    compare_figure.add_trace(
        go.Scatter(x=[xi, xi], y=[pi, yi], mode="lines",
                   line=dict(color="#16a34a", width=1.5, dash="dot"), showlegend=False)
    )
compare_figure.update_layout(
    title=f"Baseline SSE (=SST): {sst:.0f}   |   Regression SSE: {sse:.0f}",
    xaxis_title="Hours studied", yaxis_title="Exam score",
    template="plotly_white", width=750, height=500,
)
compare_figure.show()

improvement = sst - sse
fraction_removed = improvement / sst
print(f"Baseline error (SST):    {sst:.1f}")
print(f"Regression error (SSE):  {sse:.1f}")
print(f"Error removed:           {improvement:.1f}")
print(f"Fraction of error removed: {fraction_removed:.3f}")
```

</details>

In [6]:
line_x = np.linspace(scores_1d["hours_studied"].min() - 0.5, scores_1d["hours_studied"].max() + 0.5, 50)

compare_figure = go.Figure()
compare_figure.add_trace(
    go.Scatter(x=scores_1d["hours_studied"], y=scores_1d["exam_score"], mode="markers",
               marker=dict(size=11, color="#2563eb"), name="actual")
)
compare_figure.add_hline(y=baseline_prediction, line=dict(color="#f59e0b", width=3, dash="dash"))
compare_figure.add_trace(
    go.Scatter(x=[None], y=[None], mode="lines", line=dict(color="#f59e0b", width=3, dash="dash"), name="baseline (mean)")
)
compare_figure.add_trace(
    go.Scatter(x=line_x, y=slope * line_x + intercept, mode="lines",
               line=dict(color="#dc2626", width=3), name="regression line")
)
for xi, yi, pi in zip(scores_1d["hours_studied"], scores_1d["exam_score"], predictions):
    compare_figure.add_trace(
        go.Scatter(x=[xi, xi], y=[pi, yi], mode="lines",
                   line=dict(color="#16a34a", width=1.5, dash="dot"), showlegend=False)
    )
compare_figure.update_layout(
    title=f"Baseline SSE (=SST): {sst:.0f}   |   Regression SSE: {sse:.0f}",
    xaxis_title="Hours studied", yaxis_title="Exam score",
    template="plotly_white", width=750, height=500,
)
compare_figure.show()

improvement = sst - sse
fraction_removed = improvement / sst
print(f"Baseline error (SST):    {sst:.1f}")
print(f"Regression error (SSE):  {sse:.1f}")
print(f"Error removed:           {improvement:.1f}")
print(f"Fraction of error removed: {fraction_removed:.3f}")

Baseline error (SST):    2868.6
Regression error (SSE):  239.0
Error removed:           2629.5
Fraction of error removed: 0.917


That "fraction of error removed" number *is* R². Now we can write it formally:

$$R^2 = 1 - \frac{SSE}{SST} = 1 - \frac{\sum_i (y_i - \hat y_i)^2}{\sum_i (y_i - \bar y)^2}$$

- **Denominator (SST):** how wrong is the naive mean predictor?
- **Numerator (SSE):** how wrong is our regression model?

> R² asks: how much better is our regression compared with simply predicting the mean?

> ### 🙋 Ask the class
>
> - What exactly is R² comparing our model against?

## R² From Scratch, Then Sklearn

Let's compute it three ways and confirm they all agree.

<details>
<summary>Show code</summary>

```python
r2_manual = 1 - sse / sst
r2_sklearn_function = r2_score(scores_1d["exam_score"], predictions)
r2_model_score = model.score(scores_1d[["hours_studied"]], scores_1d["exam_score"])

print(f"R² (manual formula):        {r2_manual:.4f}")
print(f"R² (sklearn r2_score):      {r2_sklearn_function:.4f}")
print(f"R² (model.score):           {r2_model_score:.4f}")
```

</details>

In [7]:
r2_manual = 1 - sse / sst
r2_sklearn_function = r2_score(scores_1d["exam_score"], predictions)
r2_model_score = model.score(scores_1d[["hours_studied"]], scores_1d["exam_score"])

print(f"R² (manual formula):        {r2_manual:.4f}")
print(f"R² (sklearn r2_score):      {r2_sklearn_function:.4f}")
print(f"R² (model.score):           {r2_model_score:.4f}")

R² (manual formula):        0.9167
R² (sklearn r2_score):      0.9167
R² (model.score):           0.9167


All three match. This is a useful thing to know: for `LinearRegression`, `model.score(X, y)` **is** R² — no separate metric call needed.

## Interpretation Drills

Given an R² value, what does it actually mean in words?

### R² = 0.82

✅ Correct: "Approximately 82% of the variation in exam scores in this dataset is accounted for by the linear model using these predictors."

❌ **Wrong:** "The model is 82% accurate." R² is not an accuracy percentage — it's a variance-explained ratio.

### R² = 0.15

> Is this line useful?

The model explains relatively little variation. Whether that's still useful depends on the domain — 15% might be a huge deal in some noisy real-world settings (e.g. predicting human behavior) and disappointing in others.

### R² = -0.4

> What does a *negative* R² mean?

The model is **worse** than just predicting the mean on this evaluated data. We'll see exactly how that can happen in a moment.

> ### 🙋 Ask the class
>
> - In your own words, what does R² = 0.82 mean?
> - Is R² = 0.15 automatically a bad model?

> ### 🧑‍🏫 Instructor note
>
> The "82% accurate" misreading is extremely common and worth calling out explicitly and repeatedly —
> R² is about variance explained relative to the mean baseline, not a correctness percentage.

## A Different Question: Is This Coefficient Real?

R² tells us how well the *whole model* fits. It says nothing about whether any *one* input variable actually matters.

Let's bring back the two-feature model from last session.

<details>
<summary>Show code</summary>

```python
scores_2d = pd.read_csv("../data/study_scores_2d.csv")

model_2d = LinearRegression()
model_2d.fit(scores_2d[["hours_studied", "practice_problems"]], scores_2d["exam_score"])

coef_table_2d = pd.DataFrame({
    "feature": ["hours_studied", "practice_problems"],
    "coefficient": model_2d.coef_,
})
coef_table_2d
```

</details>

In [8]:
scores_2d = pd.read_csv("../data/study_scores_2d.csv")

model_2d = LinearRegression()
model_2d.fit(scores_2d[["hours_studied", "practice_problems"]], scores_2d["exam_score"])

coef_table_2d = pd.DataFrame({
    "feature": ["hours_studied", "practice_problems"],
    "coefficient": model_2d.coef_,
})
coef_table_2d

,feature,coefficient
0,hours_studied,4.684
1,practice_problems,1.341


Suppose `practice_problems` comes out around 0.15 — small, but not exactly zero.

> We obtained a small positive number for `practice_problems`. But is this a real effect we have convincing evidence for, or might this coefficient have appeared just because of randomness in our particular sample of students?

That's a genuinely different question from "how well does the model fit overall" — and it's exactly what hypothesis testing and p-values are for.

> ### 🙋 Ask the class
>
> - If we collected a completely different sample of students, do you think we'd get exactly the same coefficient again?

## Coefficient Uncertainty, Simulated

Here's the key intuition. Our dataset is only *one* sample. Let's simulate collecting many different samples from a relationship where the true coefficient is **exactly zero**, and see what estimated coefficients we get back.

<details>
<summary>Show code</summary>

```python
simulation_rng = np.random.default_rng(seed=99)

simulated_slopes = []
for _ in range(200):
    sample_x = simulation_rng.uniform(1, 9, size=30)
    # true relationship: y depends only on noise, NOT on x (true slope = 0)
    sample_y = 60 + simulation_rng.normal(loc=0, scale=10, size=30)

    sample_model = LinearRegression()
    sample_model.fit(sample_x.reshape(-1, 1), sample_y)
    simulated_slopes.append(sample_model.coef_[0])

simulation_figure = go.Figure()
simulation_figure.add_trace(
    go.Histogram(x=simulated_slopes, marker=dict(color="#2563eb"), nbinsx=30)
)
simulation_figure.add_vline(x=0, line=dict(color="#dc2626", width=3, dash="dash"))
simulation_figure.update_layout(
    title="Estimated Slopes Across 200 Samples (true slope = 0)",
    xaxis_title="Estimated slope", yaxis_title="Number of samples",
    template="plotly_white", width=750, height=450,
)
simulation_figure.show()

print(f"Estimated slopes ranged from {min(simulated_slopes):.2f} to {max(simulated_slopes):.2f}")
print(f"...even though the TRUE slope was exactly 0.")
```

</details>

In [9]:
simulation_rng = np.random.default_rng(seed=99)

simulated_slopes = []
for _ in range(200):
    sample_x = simulation_rng.uniform(1, 9, size=30)
    # true relationship: y depends only on noise, NOT on x (true slope = 0)
    sample_y = 60 + simulation_rng.normal(loc=0, scale=10, size=30)

    sample_model = LinearRegression()
    sample_model.fit(sample_x.reshape(-1, 1), sample_y)
    simulated_slopes.append(sample_model.coef_[0])

simulation_figure = go.Figure()
simulation_figure.add_trace(
    go.Histogram(x=simulated_slopes, marker=dict(color="#2563eb"), nbinsx=30)
)
simulation_figure.add_vline(x=0, line=dict(color="#dc2626", width=3, dash="dash"))
simulation_figure.update_layout(
    title="Estimated Slopes Across 200 Samples (true slope = 0)",
    xaxis_title="Estimated slope", yaxis_title="Number of samples",
    template="plotly_white", width=750, height=450,
)
simulation_figure.show()

print(f"Estimated slopes ranged from {min(simulated_slopes):.2f} to {max(simulated_slopes):.2f}")
print(f"...even though the TRUE slope was exactly 0.")

Estimated slopes ranged from -2.40 to 2.20
...even though the TRUE slope was exactly 0.


Even though the true relationship is exactly zero, random sampling alone produces estimated slopes scattered above and below zero — some quite far from it.

> So: how surprising is the coefficient we actually observed, *if* the true coefficient were zero? That question is exactly what a p-value answers.

## Null Hypothesis and P-Value Intuition

For a coefficient $w_j$, we set up two competing statements:

$$H_0: w_j = 0 \qquad \text{vs.} \qquad H_1: w_j \neq 0$$

> The null hypothesis $H_0$ says that, after accounting for the other predictors in the regression, this coefficient is really zero — any nonzero estimate we saw was just sampling noise.

For our example: $H_0$: the `practice_problems` coefficient is 0.

Then:

> **Assuming the null hypothesis is true**, the p-value tells us how surprising our observed result — or something even more extreme — would be.

```text
p = 0.003   ->  quite unusual under H0 (evidence against H0)
p = 0.62    ->  not particularly surprising under H0 (little evidence against H0)
```

The usual classroom threshold is `p < 0.05` — but:

> 0.05 is a commonly used convention. It is **not** a law of nature.

## What a P-Value Does NOT Mean

This is worth memorizing as a checklist.

| Claim | Verdict | Why |
| --- | :---: | --- |
| "p = 0.03 means there's a 97% probability the variable matters." | ❌ Wrong | p-values are computed *assuming* $H_0$ is true — they say nothing about the probability that $H_0$ or $H_1$ is true. |
| "p = 0.03 means there's only a 3% probability the null hypothesis is true." | ❌ Wrong | Same misconception — p-values are not probabilities of hypotheses. |
| "A small p-value means the effect is large." | ❌ Wrong | A tiny effect can have a tiny p-value if you have enough data. Statistical significance ≠ practical importance. |
| "p > 0.05 proves there is no relationship." | ❌ Wrong | It only means we didn't find strong enough evidence against $H_0$ under the model's assumptions — absence of evidence isn't evidence of absence. |

> ### 🧑‍🏫 Instructor note
>
> These four misreadings are extremely common even among practitioners — worth slowing down here and
> letting students sit with each one rather than rushing through the table.

## Why Doesn't Sklearn Just Give Us P-Values?

scikit-learn is designed mainly around predictive machine-learning workflows — fit, predict, evaluate. It doesn't compute the classical inferential statistics (standard errors, t-statistics, p-values) that come from traditional statistical regression.

For that, we reach for a different, statistics-first library: **`statsmodels`**.

## Statsmodels OLS Walkthrough

Let's fit the same two-feature model, but with `statsmodels` this time.

<details>
<summary>Show code</summary>

```python
X = scores_2d[["hours_studied", "practice_problems"]]
y = scores_2d["exam_score"]

X_with_constant = sm.add_constant(X)
results = sm.OLS(y, X_with_constant).fit()

print(results.summary())
```

</details>

In [10]:
X = scores_2d[["hours_studied", "practice_problems"]]
y = scores_2d["exam_score"]

X_with_constant = sm.add_constant(X)
results = sm.OLS(y, X_with_constant).fit()

print(results.summary())

                            OLS Regression Results                            
Dep. Variable:             exam_score   R-squared:                       0.880
Model:                            OLS   Adj. R-squared:                  0.871
Method:                 Least Squares   F-statistic:                     106.1
Date:                Thu, 27 Aug 2026   Prob (F-statistic):           4.59e-14
Time:                        18:55:29   Log-Likelihood:                -96.768
No. Observations:                  32   AIC:                             199.5
Df Residuals:                      29   BIC:                             203.9
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
                        coef    std err          t      P>|t|      [0.025      0.975]
-------------------------------------------------------------------------------------
const                30.7190      2.75

That's a lot of output. For this session, focus on just five columns: `coef`, `std err`, `t`, `P>|t|`, and the `R-squared` line near the top. Ignore the rest for now.

<details>
<summary>Show code</summary>

```python
simplified_summary = pd.DataFrame({
    "coefficient": results.params,
    "std_error": results.bse,
    "t_value": results.tvalues,
    "p_value": results.pvalues,
})
simplified_summary.index = ["Intercept", "hours studied", "practice problems"]
simplified_summary.round(4)
```

</details>

In [11]:
simplified_summary = pd.DataFrame({
    "coefficient": results.params,
    "std_error": results.bse,
    "t_value": results.tvalues,
    "p_value": results.pvalues,
})
simplified_summary.index = ["Intercept", "hours studied", "practice problems"]
simplified_summary.round(4)

,coefficient,std_error,t_value,p_value
Intercept,30.719,2.753,11.158,0.0
hours studied,4.684,0.408,11.473,0.0
practice problems,1.341,0.141,9.533,0.0


A worked interpretation, using whatever numbers your run above actually produced:

> In this synthetic dataset, holding practice problems fixed, one additional study hour corresponds to approximately `coef` additional predicted exam points, and the data provide relatively strong evidence against a zero coefficient (p well under 0.05) under the OLS assumptions.

Compare that to `practice_problems` — if its p-value is large, that's the signal that we don't have convincing evidence it matters once `hours_studied` is already in the model.

> ### 🙋 Ask the class
>
> - Looking at the table above, which coefficient has the strongest evidence against H0? Which has the weakest?

## R² and P-Values Answer Different Questions

This is the single most important idea of the session.

- **R²** asks: *how well does the model explain variation in the outcome, relative to the mean baseline?*
- **A coefficient** asks: *how much does the prediction change when this feature changes, holding the other predictors fixed?*
- **A p-value** asks: *under the null hypothesis of a zero coefficient, how surprising is the observed result?*

These are not interchangeable — and they can disagree in ways that feel surprising at first.

### Model A

```text
R² = 0.90
one feature's p-value = 0.75
```

Perfectly possible: the overall model explains a lot of variation, while that *specific* variable provides little additional evidence once the others are already accounted for.

### Model B

```text
R² = 0.10
one feature's p-value < 0.001
```

Also possible: there can be strong evidence of a real relationship for one variable, while the model as a whole still explains only a modest portion of total variation (lots of other unmeasured things are driving the outcome too).

> ### 🧑‍🏫 Instructor note
>
> If time allows, this is a good moment to pause and let the contrast sink in before moving to the guided
> practice — students often want to treat "high R²" and "significant coefficients" as the same kind of good
> news, and they are genuinely independent axes of evaluation.

## Guided Practice: The Salary Dataset

New dataset, same tools. Let's work through this one together.

In [12]:
salary_df = pd.read_csv("../data/salary_regression.csv")
salary_df.head()

,years_experience,projects_completed,coffee_cups,monthly_salary
0,1.5,1,5,1719.0
1,6.0,6,4,3386.0
2,7.2,12,3,4357.0
3,0.3,10,2,1832.0
4,1.8,12,1,2347.0


What's our target? What are our features?

> ### 🙋 Ask the class
>
> - What is the target variable here? What are the candidate features?

### Step 1 — Look at each feature against the target

<details>
<summary>Show code</summary>

```python
feature_scatter_figure = go.Figure()
colors = {"years_experience": "#2563eb", "projects_completed": "#16a34a", "coffee_cups": "#dc2626"}
for feature, color in colors.items():
    feature_scatter_figure = go.Figure()
    feature_scatter_figure.add_trace(
        go.Scatter(x=salary_df[feature], y=salary_df["monthly_salary"], mode="markers",
                   marker=dict(size=9, color=color))
    )
    feature_scatter_figure.update_layout(
        title=f"{feature} vs monthly_salary",
        xaxis_title=feature, yaxis_title="monthly_salary",
        template="plotly_white", width=500, height=350,
    )
    feature_scatter_figure.show()
```

</details>

In [13]:
for feature, color in {"years_experience": "#2563eb", "projects_completed": "#16a34a", "coffee_cups": "#dc2626"}.items():
    feature_scatter_figure = go.Figure()
    feature_scatter_figure.add_trace(
        go.Scatter(x=salary_df[feature], y=salary_df["monthly_salary"], mode="markers",
                   marker=dict(size=9, color=color))
    )
    feature_scatter_figure.update_layout(
        title=f"{feature} vs monthly_salary",
        xaxis_title=feature, yaxis_title="monthly_salary",
        template="plotly_white", width=500, height=350,
    )
    feature_scatter_figure.show()

> ### 🙋 Ask the class
>
> - Just from these scatter plots, which variables look like they matter? Which one looks like noise?

### Step 2 — Fit with sklearn

In [14]:
salary_features = ["years_experience", "projects_completed", "coffee_cups"]

X_salary = salary_df[salary_features]
y_salary = salary_df["monthly_salary"]

salary_model = LinearRegression()
salary_model.fit(X_salary, y_salary)

salary_coef_df = pd.DataFrame({
    "feature": salary_features,
    "coefficient": salary_model.coef_,
})
salary_coef_df

,feature,coefficient
0,years_experience,355.289
1,projects_completed,51.243
2,coffee_cups,-7.534


### Step 3 — R², three ways

In [15]:
salary_predictions = salary_model.predict(X_salary)
salary_residuals = y_salary - salary_predictions
salary_sse = np.sum(salary_residuals ** 2)

salary_baseline = np.full(len(y_salary), y_salary.mean())
salary_sst = np.sum((y_salary - salary_baseline) ** 2)

salary_r2_manual = 1 - salary_sse / salary_sst
salary_r2_sklearn = salary_model.score(X_salary, y_salary)

print(f"R² (manual):  {salary_r2_manual:.4f}")
print(f"R² (sklearn): {salary_r2_sklearn:.4f}")

R² (manual):  0.9656
R² (sklearn): 0.9656


### Step 4 — Statsmodels: which coefficients have real evidence?

In [16]:
X_salary_sm = sm.add_constant(X_salary)
salary_results = sm.OLS(y_salary, X_salary_sm).fit()

salary_summary_table = pd.DataFrame({
    "coefficient": salary_results.params,
    "std_error": salary_results.bse,
    "p_value": salary_results.pvalues,
})
salary_summary_table.round(4)

,coefficient,std_error,p_value
const,1145.688,98.545,0.000
years_experience,355.289,9.950,0.000
projects_completed,51.243,7.456,0.000
coffee_cups,-7.534,20.518,0.715


> ### 🙋 Ask the class
>
> - Which variable has the strongest coefficient?
> - Which variables have convincing evidence against coefficient = 0?
> - Does coffee consumption appear useful?
> - What does the overall R² tell us here?

> ### 🧑‍🏫 Instructor note
>
> Expect: years_experience and projects_completed should both come back with small p-values (real signal),
> while coffee_cups should come back with a large p-value (no real relationship) — that contrast is the
> entire point of this dataset. Run this live and let the numbers make the case rather than asserting it.

## What If the Outcome Isn't a Number?

Everything so far has predicted a continuous number: an exam score, a salary. What if instead we want to predict something like **pass or fail**?

In [17]:
pass_df = pd.read_csv("../data/exam_pass_classification.csv")
pass_df

,hours_studied,passed
0,1.0,0
1,2.0,0
2,2.5,0
3,3.0,0
4,4.0,0
5,4.5,1
6,5.0,1
7,6.0,1
8,7.0,1
9,8.0,1


```text
0 = failed
1 = passed
```

> Can we use our normal linear regression here? Let's just try it.

<details>
<summary>Show code</summary>

```python
pass_model = LinearRegression()
pass_model.fit(pass_df[["hours_studied"]], pass_df["passed"])

line_x = np.linspace(0, 9, 100)
line_y = pass_model.coef_[0] * line_x + pass_model.intercept_

pass_figure = go.Figure()
pass_figure.add_trace(
    go.Scatter(x=pass_df["hours_studied"], y=pass_df["passed"], mode="markers",
               marker=dict(size=13, color="#2563eb"), name="actual (0 or 1)")
)
pass_figure.add_trace(
    go.Scatter(x=line_x, y=line_y, mode="lines", line=dict(color="#dc2626", width=3), name="linear regression fit")
)
pass_figure.add_hline(y=1, line=dict(color="gray", width=1, dash="dot"))
pass_figure.add_hline(y=0, line=dict(color="gray", width=1, dash="dot"))
pass_figure.update_layout(
    title="Fitting an Ordinary Line to a 0/1 Outcome",
    xaxis_title="Hours studied", yaxis_title="Passed (0 or 1)",
    template="plotly_white", width=750, height=500,
)
pass_figure.show()

print(f"Predicted value at 8.5 hours: {pass_model.predict([[8.5]])[0]:.2f}")
print(f"Predicted value at 0.5 hours: {pass_model.predict([[0.5]])[0]:.2f}")
```

</details>

In [18]:
pass_model = LinearRegression()
pass_model.fit(pass_df[["hours_studied"]], pass_df["passed"])

line_x = np.linspace(0, 9, 100)
line_y = pass_model.coef_[0] * line_x + pass_model.intercept_

pass_figure = go.Figure()
pass_figure.add_trace(
    go.Scatter(x=pass_df["hours_studied"], y=pass_df["passed"], mode="markers",
               marker=dict(size=13, color="#2563eb"), name="actual (0 or 1)")
)
pass_figure.add_trace(
    go.Scatter(x=line_x, y=line_y, mode="lines", line=dict(color="#dc2626", width=3), name="linear regression fit")
)
pass_figure.add_hline(y=1, line=dict(color="gray", width=1, dash="dot"))
pass_figure.add_hline(y=0, line=dict(color="gray", width=1, dash="dot"))
pass_figure.update_layout(
    title="Fitting an Ordinary Line to a 0/1 Outcome",
    xaxis_title="Hours studied", yaxis_title="Passed (0 or 1)",
    template="plotly_white", width=750, height=500,
)
pass_figure.show()

print(f"Predicted value at 8.5 hours: {pass_model.predict([[8.5]])[0]:.2f}")
print(f"Predicted value at 0.5 hours: {pass_model.predict([[0.5]])[0]:.2f}")

Predicted value at 8.5 hours: 1.33
Predicted value at 0.5 hours: -0.25


/Users/abdukarimov/workspaces/work/humblebeeai/int.academy-tutorial/linear-regression/.venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(
/Users/abdukarimov/workspaces/work/humblebeeai/int.academy-tutorial/linear-regression/.venv/lib/python3.9/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LinearRegression was fitted with feature names
  warnings.warn(


Look at the line — it pokes above 1 on the right and below 0 on the left.

> What does a prediction of 1.15 mean? What does a prediction of -0.08 mean?

Neither is a sensible probability. This isn't a bug in our code — it's a fundamental mismatch between an unbounded straight line and an outcome that can only ever be 0 or 1.

> ### 🙋 Ask the class
>
> - What does it mean for a model to predict 1.15 for a pass/fail outcome?

## Setting Up What Comes Next

What we actually need is a model where:

$$0 \leq \hat p \leq 1$$

and where:

```text
small x  -> probability close to 0
middle x -> rapid transition
large x  -> probability close to 1
```

That S-shaped curve is called the **sigmoid function**.

<details>
<summary>Show code</summary>

```python
sigmoid_x = np.linspace(-10, 10, 200)
sigmoid_y = 1 / (1 + np.exp(-sigmoid_x))

sigmoid_figure = go.Figure()
sigmoid_figure.add_trace(
    go.Scatter(x=sigmoid_x, y=sigmoid_y, mode="lines", line=dict(color="#7c3aed", width=3))
)
sigmoid_figure.update_layout(
    title="The Sigmoid Function",
    xaxis_title="input", yaxis_title="probability",
    template="plotly_white", width=650, height=450,
)
sigmoid_figure.show()
```

</details>

In [19]:
sigmoid_x = np.linspace(-10, 10, 200)
sigmoid_y = 1 / (1 + np.exp(-sigmoid_x))

sigmoid_figure = go.Figure()
sigmoid_figure.add_trace(
    go.Scatter(x=sigmoid_x, y=sigmoid_y, mode="lines", line=dict(color="#7c3aed", width=3))
)
sigmoid_figure.update_layout(
    title="The Sigmoid Function",
    xaxis_title="input", yaxis_title="probability",
    template="plotly_white", width=650, height=450,
)
sigmoid_figure.show()

```text
Linear Regression:
features -> weighted sum -> any real-valued prediction

Logistic Regression:
features -> weighted sum -> sigmoid -> 0-1 probability
```

We won't go further into logistic regression today — that's next session's whole topic.

> **Next time:** instead of predicting a continuous number like exam score, we'll predict the **probability of belonging to a class**.

## Recap

```text
SSE       -> what the optimizer minimizes
R²        -> how good the fit is, relative to predicting the mean
p-value   -> how much statistical evidence we have for one coefficient
```

These are three different questions. A model can have high R² with an unconvincing individual coefficient, or a convincing individual coefficient inside a model with modest overall R². Understanding that distinction is exactly what puts you in a good position to move into logistic regression next session.